UN Debates - DETM

In [2]:
import pickle
import numpy as np
from scipy.io import loadmat
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

ck = 'datasets/un_DETM/min_df_100/detm_undebates_K_50_Htheta_800_Optim_adam_Clip_0.0_ThetaAct_relu_Lr_0.005_Bsz_1000_RhoSize_300_L_3_minDF_100_trainEmbeddings_1'
rho = loadmat(ck + '_rho.mat')['values']

ckpt = 'datasets/un_DETM/min_df_100/detm_undebates_K_50_Htheta_800_Optim_adam_Clip_0.0_ThetaAct_relu_Lr_0.005_Bsz_1000_RhoSize_300_L_3_minDF_100_trainEmbeddings_1'
beta = loadmat(ckpt + '_beta.mat')['values']  # K x T x V

K, T, V = beta.shape
_, L_rho = rho.shape

topic_embs = np.zeros((K, T, L_rho))

for t in range(T):
    for k in range(K):
        rho_norm = normalize(rho, axis=1)

        topic_embs[k, t, :] = beta[k, t, :] @ rho_norm

flat = topic_embs.reshape(-1, L_rho)
flat_norm = normalize(flat, axis=1)  # unit vectors
topic_embs = flat_norm.reshape(K, T, L_rho)

edges = []

for t in range(T - 1):
    for k in range(K):
        emb_t = topic_embs[k, t, :].reshape(1, -1)
        emb_t1 = topic_embs[k, t + 1, :].reshape(1, -1)

        sim = cosine_similarity(emb_t, emb_t1)[0, 0]

        edges.append({
            "relation": "continued",
            "similarity": sim,
            "window1": t,
            "window2": t + 1,
            "topic_id": k
        })

detm_edges_df = pd.DataFrame(edges)

#detm_edges_df.to_csv("datasets/un_detm_relations.csv", index=False)

In [3]:
detm_edges_df

,relation,similarity,window1,window2,topic_id
0,continued,0.118301,0,1,0
1,continued,0.096463,0,1,1
2,continued,0.203414,0,1,2
3,continued,0.038309,0,1,3
4,continued,0.182514,0,1,4
...,...,...,...,...,...
2245,continued,0.053454,44,45,45
2246,continued,0.050543,44,45,46
2247,continued,0.154911,44,45,47
2248,continued,-0.028105,44,45,48


State of Union - DETM

In [ ]:
import pickle
import numpy as np
from scipy.io import loadmat

ck = 'datasets/stateofunion_DETM/min_df_10/detm_stateofunion_K_50_Htheta_800_Optim_adam_Clip_0.0_ThetaAct_relu_Lr_0.005_Bsz_1000_RhoSize_300_L_3_minDF_10_trainEmbeddings_1'
rho = loadmat(ck + '_rho.mat')['values']

ckpt = 'datasets/stateofunion_DETM/min_df_10/detm_stateofunion_K_50_Htheta_800_Optim_adam_Clip_0.0_ThetaAct_relu_Lr_0.005_Bsz_1000_RhoSize_300_L_3_minDF_10_trainEmbeddings_1'
beta = loadmat(ckpt + '_beta.mat')['values']  # K x T x V

K, T, V = beta.shape
_, L_rho = rho.shape

topic_embs = np.zeros((K, T, L_rho))

for t in range(T):
    for k in range(K):
        topic_embs[k, t, :] = beta[k, t, :] @ rho  

flat = topic_embs.reshape(-1, L_rho)
flat_norm = normalize(flat, axis=1)  # unit vectors
topic_embs = flat_norm.reshape(K, T, L_rho)

edges = []

for t in range(T - 1):
    for k in range(K):
        emb_t = topic_embs[k, t, :].reshape(1, -1)
        emb_t1 = topic_embs[k, t + 1, :].reshape(1, -1)

        sim = cosine_similarity(emb_t, emb_t1)[0, 0]

        edges.append({
            "relation": "continued",
            "similarity": sim,
            "window1": t,
            "window2": t + 1,
            "topic_id": k
        })

detm_edges_df = pd.DataFrame(edges)

detm_edges_df.to_csv("../../datasets/stateofunion/stateofunion_detm_relations.csv", index=False)